# Notebook 04 — PyMC Fundamentals: Building Bayesian Models

## Background

Notebook 03 showed you the machinery behind NUTS and why it works. Now we slow down and learn how to actually *build* models in PyMC — the tool you'll use for the rest of this series. Getting fluent with PyMC's model-building API is the difference between understanding Bayesian ideas conceptually and being able to apply them to real data.

PyMC uses a context-manager syntax (`with pm.Model() as model:`) to define probabilistic models. Inside the context, you declare random variables as distributions — PyMC tracks the log-probability automatically and passes `grad log P(theta|data)` to NUTS. You can combine distributions arbitrarily, add deterministic transformations, and let PyMC figure out how to sample.

## What You'll Learn

- The PyMC model context and variable types: free RVs, `Deterministic`, `Data`, observed
- Prior predictive checks: does your prior make sense before seeing data?
- The full Bayesian workflow: prior -> sample -> diagnose -> posterior predictive
- `pm.Data` for out-of-sample predictions without redefining the model
- `pm.Deterministic` for derived quantities with full uncertainty propagation
- Choosing weakly informative priors in practice
- Robust likelihoods: Student-t when Normal fails

## Why This Matters

Most PyMC bugs aren't sampling failures — they're model specification bugs that NUTS happily samples from, returning a wrong but plausible-looking posterior. Prior predictive checks catch these before you spend compute time on sampling. The `pm.Data` pattern matters for deployment: you define the model once and swap in new observations at prediction time. Understanding `pm.Deterministic` matters for inference — it's how you compute derived quantities (odds ratios, effect sizes, predictions) and include them in the trace with full uncertainty.

In [ ]:
import pymc as pm
import arviz as az
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

print(f'PyMC {pm.__version__}  |  ArviZ {az.__version__}')

## Part 1 — The PyMC Model Context

Every PyMC model lives inside a `with pm.Model()` block. Variables declared inside are automatically registered. PyMC tracks:

- **Free variables** (`pm.Normal`, `pm.Beta`, etc.): parameters to sample
- **Observed variables** (`observed=` kwarg): fixed to data; contribute to the log-likelihood
- **Deterministic variables** (`pm.Deterministic`): computed quantities that appear in the trace
- **Data containers** (`pm.Data`): mutable data that can be swapped at prediction time

The model object exposes `model.logp()`, `model.compile_logp()`, `model.free_RVs`, and `model.initial_point()` — all useful for debugging and understanding what PyMC is doing internally.

In [ ]:
# Simplest model: unknown mean and scale of a Normal distribution
np.random.seed(42)
true_mu, true_sigma = 5.3, 1.2
observations = np.random.normal(true_mu, true_sigma, size=20)

with pm.Model() as normal_model:
    mu = pm.Normal('mu', mu=0, sigma=10)       # prior: vague belief about mean
    sigma = pm.HalfNormal('sigma', sigma=5)    # prior: sigma must be positive
    y_obs = pm.Normal('y_obs', mu=mu, sigma=sigma, observed=observations)

print('Model structure:')
print(f'  Free variables:     {[v.name for v in normal_model.free_RVs]}')
print(f'  Observed variables: {[v.name for v in normal_model.observed_RVs]}')
print(f'  Initial point:      {normal_model.initial_point()}')

# Log-probability at different parameter points
logp_fn = normal_model.compile_logp()
lp_true  = logp_fn({'mu': true_mu,  'sigma_log__': np.log(true_sigma)})
lp_wrong = logp_fn({'mu': 0.0,      'sigma_log__': np.log(5.0)})
print(f'\nlog P(theta|data) at true params:  {lp_true:.2f}')
print(f'log P(theta|data) at wrong params: {lp_wrong:.2f}')
print('(Higher = more consistent with data and priors)')

## Part 2 — Prior Predictive Checks

Before sampling the posterior, ask: *if we draw from the prior and simulate data, do the predictions look plausible?*

This catches prior-data conflicts early. If the prior predictive covers ranges that are physically impossible for your problem (negative ages, probabilities > 1, etc.) or completely excludes the observed data range, the model has a bug before NUTS ever runs.

In [ ]:
with normal_model:
    prior_pred = pm.sample_prior_predictive(samples=500, random_seed=42)

prior_mu_samp    = prior_pred.prior['mu'].values.flatten()
prior_sigma_samp = prior_pred.prior['sigma'].values.flatten()
prior_y_samp     = prior_pred.prior_predictive['y_obs'].values.flatten()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(prior_mu_samp, bins=50, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].axvline(true_mu, color='red', linestyle='--', label=f'True mu={true_mu}')
axes[0].set_xlabel('mu'); axes[0].set_title('Prior on mu: Normal(0, 10)')
axes[0].legend()

axes[1].hist(prior_sigma_samp, bins=50, color='darkorange', alpha=0.7, edgecolor='white')
axes[1].axvline(true_sigma, color='red', linestyle='--', label=f'True sigma={true_sigma}')
axes[1].set_xlabel('sigma'); axes[1].set_title('Prior on sigma: HalfNormal(5)')
axes[1].legend()

axes[2].hist(prior_y_samp, bins=60, color='seagreen', alpha=0.7, density=True, edgecolor='white')
axes[2].hist(observations, bins=12, color='red', alpha=0.5, density=True,
             edgecolor='white', label='Observed data')
axes[2].set_xlabel('y'); axes[2].set_title('Prior Predictive vs. Observed Data')
axes[2].legend()

plt.suptitle('Prior Predictive Check -- Run This Before pm.sample()', fontsize=12)
plt.tight_layout()
plt.show()

print(f'Prior predictive y 94% range: [{np.percentile(prior_y_samp, 3):.1f}, {np.percentile(prior_y_samp, 97):.1f}]')
print(f'Actual data range:             [{observations.min():.1f}, {observations.max():.1f}]')
print('Observed data falls within prior predictive range -- no prior-data conflict.')

In [ ]:
# Sample the posterior and compare prior vs posterior
with normal_model:
    trace_normal = pm.sample(draws=2000, tune=1000, chains=4,
                             random_seed=42, progressbar=True)

print(az.summary(trace_normal, var_names=['mu', 'sigma']).to_string())
print(f'\nTrue: mu={true_mu}, sigma={true_sigma}')
print(f'MLE:  mu={observations.mean():.3f}, sigma={observations.std():.3f}')

In [ ]:
# Prior-to-posterior shift visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, var, prior_samp, true_val in zip(
    axes,
    ['mu', 'sigma'],
    [prior_mu_samp, prior_sigma_samp],
    [true_mu, true_sigma]
):
    post_samp = trace_normal.posterior[var].values.flatten()
    
    # Prior KDE (clipped for readability)
    samp_clip = prior_samp[(prior_samp > post_samp.min() - 2) & (prior_samp < post_samp.max() + 2)]
    if len(samp_clip) > 10:
        xr = np.linspace(post_samp.min() - 1, post_samp.max() + 1, 300)
        kde_prior = stats.gaussian_kde(samp_clip)
        ax.plot(xr, kde_prior(xr), 'gray', lw=1.5, linestyle='--', alpha=0.7, label='Prior')
    
    # Posterior
    xp = np.linspace(post_samp.min() - 0.3, post_samp.max() + 0.3, 300)
    kde_post = stats.gaussian_kde(post_samp)
    ax.fill_between(xp, kde_post(xp), alpha=0.3, color='steelblue')
    ax.plot(xp, kde_post(xp), 'steelblue', lw=2, label='Posterior')
    ax.axvline(true_val, color='red', linestyle='--', lw=1.5, label=f'True: {true_val}')
    ax.set_xlabel(var); ax.set_title(f'Prior -> Posterior: {var}')
    ax.legend()

plt.suptitle('Bayesian Updating: 20 Observations Sharpens the Posterior', fontsize=12)
plt.tight_layout()
plt.show()

## Part 3 — pm.Data: Mutable Data Containers

`pm.Data` registers a data array with the model but keeps it *mutable* — you swap it later using `pm.set_data()`. This is essential for:
- **Out-of-sample prediction**: fit on training data, predict on test inputs without rebuilding the model
- **Online updating**: add new observations without reconstructing the graph
- **Sensitivity analysis**: swap data slices to test robustness

In [ ]:
np.random.seed(42)
n_train = 40
x_train = np.random.uniform(0, 10, n_train)
y_train = 2.5 * x_train - 1.0 + np.random.normal(0, 2.0, n_train)
x_test  = np.array([0.5, 2.0, 5.0, 8.0, 9.5])

with pm.Model() as regression_model:
    x_data = pm.Data('x', x_train)               # mutable data container
    alpha  = pm.Normal('alpha', mu=0, sigma=10)
    beta   = pm.Normal('beta',  mu=0, sigma=5)
    sigma  = pm.HalfNormal('sigma', sigma=5)
    mu_pred = pm.Deterministic('mu_pred', alpha + beta * x_data)
    y_obs   = pm.Normal('y_obs', mu=mu_pred, sigma=sigma, observed=y_train)
    
    trace_reg = pm.sample(draws=1000, tune=500, chains=4,
                          random_seed=42, progressbar=False)

print('Fitted regression:')
print(az.summary(trace_reg, var_names=['alpha', 'beta', 'sigma']).to_string())
print(f'\nTrue: alpha=-1.0, beta=2.5, sigma=2.0')

In [ ]:
# Predict on test data by swapping pm.Data -- no model rebuild needed
with regression_model:
    pm.set_data({'x': x_test})
    pred_test = pm.sample_posterior_predictive(
        trace_reg, var_names=['mu_pred', 'y_obs'], random_seed=42
    )

mu_pred_samp = pred_test.posterior_predictive['mu_pred'].values.reshape(-1, len(x_test))
y_pred_samp  = pred_test.posterior_predictive['y_obs'].values.reshape(-1, len(x_test))
true_y_test  = 2.5 * x_test - 1.0

print('Test predictions:')
print(f'{"x":>6} {"True y":>8} {"Pred mean":>10} {"94% PI":>22}')
print('-' * 52)
for i, x in enumerate(x_test):
    mean_pred = mu_pred_samp[:, i].mean()
    lo, hi = np.percentile(y_pred_samp[:, i], [3, 97])
    print(f'{x:>6.1f} {true_y_test[i]:>8.2f} {mean_pred:>10.2f} ({lo:6.2f}, {hi:6.2f})')

In [ ]:
# Visualize fit with predictive intervals
x_plot = np.linspace(0, 10, 200)

with regression_model:
    pm.set_data({'x': x_plot})
    pp_plot = pm.sample_posterior_predictive(
        trace_reg, var_names=['mu_pred', 'y_obs'], random_seed=42
    )

mu_plot = pp_plot.posterior_predictive['mu_pred'].values.reshape(-1, len(x_plot))
y_plot  = pp_plot.posterior_predictive['y_obs'].values.reshape(-1, len(x_plot))

fig, ax = plt.subplots(figsize=(10, 6))
ax.fill_between(x_plot,
                np.percentile(y_plot, 3, axis=0),
                np.percentile(y_plot, 97, axis=0),
                alpha=0.15, color='steelblue', label='94% predictive interval')
ax.fill_between(x_plot,
                np.percentile(mu_plot, 3, axis=0),
                np.percentile(mu_plot, 97, axis=0),
                alpha=0.3, color='steelblue', label='94% mean CI')
ax.plot(x_plot, mu_plot.mean(axis=0), 'steelblue', lw=2, label='Posterior mean')
ax.plot(x_plot, 2.5 * x_plot - 1.0, 'r--', lw=1.5, label='True line')
ax.scatter(x_train, y_train, color='steelblue', alpha=0.4, s=30, zorder=3, label='Training data')
ax.scatter(x_test, true_y_test, color='red', s=100, zorder=5, marker='*', label='Test points')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Bayesian Linear Regression with pm.Data\n'
             'Predictive interval includes parameter + observation uncertainty')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Part 4 — pm.Deterministic: Derived Quantities with Full Uncertainty

`pm.Deterministic` wraps any expression of model variables and adds it to the posterior trace. This is how you compute effect sizes, predictions, or any derived quantity you want to quantify uncertainty over — all with the same MCMC samples, no post-hoc computation needed.

In [ ]:
# Logistic regression with clinically meaningful derived quantities
np.random.seed(42)
n = 200
age = np.random.normal(50, 12, n)
treatment = np.random.binomial(1, 0.5, n)
log_odds_true = -3.0 + 0.05 * age - 0.8 * treatment
disease = np.random.binomial(1, 1 / (1 + np.exp(-log_odds_true)))

# Standardize age for better geometry
age_mean, age_std = age.mean(), age.std()
age_z = (age - age_mean) / age_std

with pm.Model() as disease_model:
    intercept  = pm.Normal('intercept',  mu=0, sigma=2)
    beta_age   = pm.Normal('beta_age',   mu=0, sigma=1)
    beta_trt   = pm.Normal('beta_trt',   mu=0, sigma=1)
    
    logit_p = intercept + beta_age * age_z + beta_trt * treatment
    
    # Derived quantities tracked in the trace
    age_50_z = (50 - age_mean) / age_std
    age_60_z = (60 - age_mean) / age_std
    
    OR_treatment   = pm.Deterministic('OR_treatment',   pm.math.exp(beta_trt))
    risk_50_ctrl   = pm.Deterministic('risk_50_ctrl',   pm.math.sigmoid(intercept + beta_age * age_50_z))
    risk_50_trt    = pm.Deterministic('risk_50_trt',    pm.math.sigmoid(intercept + beta_age * age_50_z + beta_trt))
    ARR_50         = pm.Deterministic('ARR_50',         risk_50_ctrl - risk_50_trt)
    
    y = pm.Bernoulli('y', logit_p=logit_p, observed=disease)
    
    trace_disease = pm.sample(draws=2000, tune=1000, chains=4,
                               random_seed=42, progressbar=False)

derived = ['OR_treatment', 'risk_50_ctrl', 'risk_50_trt', 'ARR_50']
print('Posterior Summary (parameters + derived quantities):')
print(az.summary(trace_disease, var_names=['intercept', 'beta_age', 'beta_trt'] + derived).to_string())
print(f'\nTrue OR_treatment = {np.exp(-0.8):.3f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Odds ratio
or_samp = trace_disease.posterior['OR_treatment'].values.flatten()
axes[0].hist(or_samp, bins=60, color='steelblue', alpha=0.7, edgecolor='white', density=True)
axes[0].axvline(1.0, color='black', linestyle='--', lw=1, label='OR=1 (no effect)')
axes[0].axvline(np.exp(-0.8), color='red', linestyle='--', lw=1.5, label=f'True: {np.exp(-0.8):.2f}')
lo, hi = np.percentile(or_samp, [3, 97])
axes[0].axvspan(lo, hi, alpha=0.15, color='steelblue', label=f'94% CI: [{lo:.2f}, {hi:.2f}]')
axes[0].set_xlabel('Odds Ratio'); axes[0].set_title('Treatment Odds Ratio')
axes[0].legend(fontsize=8)

# Risk at age 50
r50c = trace_disease.posterior['risk_50_ctrl'].values.flatten()
r50t = trace_disease.posterior['risk_50_trt'].values.flatten()
axes[1].hist(r50c, bins=50, alpha=0.6, color='steelblue', density=True, label='Control')
axes[1].hist(r50t, bins=50, alpha=0.6, color='darkorange', density=True, label='Treated')
axes[1].set_xlabel('P(disease)'); axes[1].set_title('Disease Risk at Age 50')
axes[1].legend()

# Absolute risk reduction
arr_samp = trace_disease.posterior['ARR_50'].values.flatten()
axes[2].hist(arr_samp, bins=60, color='seagreen', alpha=0.7, edgecolor='white', density=True)
axes[2].axvline(0, color='black', linestyle='--', lw=1)
lo2, hi2 = np.percentile(arr_samp, [3, 97])
axes[2].axvspan(lo2, hi2, alpha=0.15, color='seagreen')
axes[2].set_xlabel('ARR'); axes[2].set_title(f'Absolute Risk Reduction at 50\n94% CI: [{lo2:.3f}, {hi2:.3f}]')

plt.suptitle('pm.Deterministic: Derived Quantities with Full Posterior Uncertainty', fontsize=12)
plt.tight_layout()
plt.show()

## Part 5 — Posterior Predictive Checks

**Prior predictive**: does the prior make sense?
**Posterior predictive check (PPC)**: does the *fitted model* generate data that looks like the observed data?

PPCs simulate new datasets from the posterior and compare them to observations on specific test statistics. If the model can't reproduce the observed mean, variance, or tail behavior, it's misspecified — not just uncertain, but *wrong in a systematic way*.

In [ ]:
# Data with heavy tails -- Normal likelihood will underestimate the tails
np.random.seed(42)
n_obs = 150
data = np.concatenate([
    np.random.normal(5, 1, 120),
    np.random.normal(5, 6, 30)  # heavy tail component
])

with pm.Model() as normal_likelihood_model:
    mu    = pm.Normal('mu',    mu=5, sigma=5)
    sigma = pm.HalfNormal('sigma', sigma=5)
    y_obs = pm.Normal('y_obs', mu=mu, sigma=sigma, observed=data)
    trace_norm  = pm.sample(draws=1000, tune=500, chains=2, random_seed=42, progressbar=False)
    ppc_normal  = pm.sample_posterior_predictive(trace_norm, random_seed=42)

y_rep = ppc_normal.posterior_predictive['y_obs'].values.reshape(-1, n_obs)

# Compare test statistics
test_funcs = {
    'mean':   np.mean,
    'std':    np.std,
    '95th pct': lambda x: np.percentile(x, 95),
    '5th pct':  lambda x: np.percentile(x, 5),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (stat_name, fn) in zip(axes, test_funcs.items()):
    rep_vals = [fn(y_rep[i]) for i in range(len(y_rep))]
    obs_val  = fn(data)
    p_bayesian = np.mean(np.array(rep_vals) > obs_val)
    ax.hist(rep_vals, bins=40, color='steelblue', alpha=0.7, edgecolor='white', density=True)
    ax.axvline(obs_val, color='red', lw=2, linestyle='--', label=f'Obs: {obs_val:.2f}')
    ax.set_xlabel(stat_name)
    ax.set_title(f'PPC: {stat_name}\nBayesian p = {p_bayesian:.2f}')
    ax.legend(fontsize=8)

plt.suptitle('PPC: Normal Likelihood on Heavy-Tailed Data\n'
             'p near 0 or 1 = model fails this statistic', fontsize=12)
plt.tight_layout()
plt.show()

print('The tail statistics (5th/95th percentiles) reveal the misspecification.')
print('The Normal model cannot reproduce the observed extremes.')

In [ ]:
# Fix: Student-t likelihood for robust inference
with pm.Model() as student_t_model:
    mu    = pm.Normal('mu',    mu=5, sigma=5)
    sigma = pm.HalfNormal('sigma', sigma=5)
    nu    = pm.Exponential('nu', lam=1/30)  # degrees of freedom: most mass on 1-30
    y_obs = pm.StudentT('y_obs', nu=nu, mu=mu, sigma=sigma, observed=data)
    trace_t = pm.sample(draws=1000, tune=500, chains=4, random_seed=42, progressbar=False)
    ppc_t   = pm.sample_posterior_predictive(trace_t, random_seed=42)

print('Student-t model summary:')
print(az.summary(trace_t, var_names=['mu', 'sigma', 'nu']).to_string())
print('\nnu close to small values = heavy tails detected by the model')

In [ ]:
# ArviZ built-in PPC plot: Normal vs Student-t
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (ppc, title) in zip(axes, [
    (ppc_normal, 'Normal Likelihood (misspecified)'),
    (ppc_t,      'Student-t Likelihood (robust)'),
]):
    az.plot_ppc(ppc, observed=True, num_pp_samples=80, ax=ax)
    ax.set_title(title)

plt.suptitle('PPC Comparison: Gray=replicated, Black=observed', fontsize=12)
plt.tight_layout()
plt.show()

## Part 6 — Choosing Weakly Informative Priors

PyMC gives you complete control over priors. The right prior is model-specific, but there are reliable defaults:

In [ ]:
# Common weakly informative prior choices
import warnings; warnings.filterwarnings('ignore')

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
np.random.seed(42)

prior_configs = [
    ('Normal(0, 1)', 'Standardized regression coefficients\n(after centering predictors)',
     np.random.normal(0, 1, 5000), np.linspace(-4, 4, 300), 'steelblue'),
    ('Normal(0, 5)', 'Regression coefficients\n(weakly informative, unscaled)',
     np.random.normal(0, 5, 5000), np.linspace(-15, 15, 300), 'steelblue'),
    ('HalfNormal(1)', 'Scale parameters (sigma, tau)\nForced positive, mode at 0',
     abs(np.random.normal(0, 1, 5000)), np.linspace(0, 5, 300), 'darkorange'),
    ('HalfNormal(5)', 'Scale with wide uncertainty',
     abs(np.random.normal(0, 5, 5000)), np.linspace(0, 20, 300), 'darkorange'),
    ('Exponential(1/30)', 'Degrees of freedom (nu)\nMost mass on realistic range',
     np.random.exponential(30, 5000), np.linspace(0, 80, 300), 'seagreen'),
    ('Beta(1, 1)', 'Probabilities\nUniform over [0, 1]',
     np.random.beta(1, 1, 5000), np.linspace(0, 1, 300), 'crimson'),
]

for ax, (name, desc, samp, xr, color) in zip(axes.flatten(), prior_configs):
    kde = stats.gaussian_kde(samp)
    ax.plot(xr, kde(xr), color=color, lw=2)
    ax.fill_between(xr, kde(xr), alpha=0.2, color=color)
    lo, hi = np.percentile(samp, [3, 97])
    ax.axvspan(lo, hi, alpha=0.12, color='gray')
    ax.set_title(f'{name}\n{desc}', fontsize=9)
    ax.set_xlabel('Value'); ax.set_ylabel('Density')
    ax.text(0.97, 0.88, f'94%: [{lo:.1f}, {hi:.1f}]',
            transform=ax.transAxes, ha='right', fontsize=7.5, color='#555')

plt.suptitle('Common Weakly Informative Priors\nGray band = 94% prior interval', fontsize=12)
plt.tight_layout()
plt.show()

print('Prior selection heuristics:')
print('  - Regression coefficients (standardized predictors): Normal(0, 1)')
print('  - Standard deviations: HalfNormal(1) or Exponential(1)')
print('  - Probabilities: Beta(1,1) = uniform; Beta(2,2) = gently regularizing')
print('  - Student-t degrees of freedom: Exponential(1/30)')
print('  - Always run prior predictive to verify the prior is sensible')

## Summary

The full PyMC workflow:

```python
with pm.Model() as model:
    x_data = pm.Data('x', x_train)                      # mutable data
    alpha  = pm.Normal('alpha', mu=0, sigma=10)          # prior
    mu     = pm.Deterministic('mu', alpha + beta * x_data)  # derived quantity
    y_obs  = pm.Normal('y_obs', mu=mu, sigma=sigma, observed=y_train)  # likelihood
    
    prior_pred = pm.sample_prior_predictive()            # check BEFORE sampling
    trace      = pm.sample(draws=2000, tune=1000, chains=4)  # NUTS
    post_pred  = pm.sample_posterior_predictive(trace)   # model fit check
    
    pm.set_data({'x': x_test})                          # swap for prediction
    pred       = pm.sample_posterior_predictive(trace)  # out-of-sample
```

**Key API**:
- `pm.Data` — mutable; use for out-of-sample prediction
- `pm.Deterministic` — computed quantities appear in trace with full uncertainty
- `pm.sample_prior_predictive` — always do this before `pm.sample()`
- `az.plot_ppc` — model misspecification shows up here, not in R-hat
- Student-t likelihood — drop-in for Normal when data has heavy tails

**Next**: Notebook 05 — Hierarchical Models. Partial pooling, the non-centered parameterization (curing Neal's funnel), and why multilevel models dramatically outperform per-group estimates.